# NB07 — Train/Validation/Test Split Materialisation

**Project:** Evolutionary Computation for Sepsis Mortality Prediction
**Input:** `data/processed/features_curated.parquet` (11,164 × 61), `data/processed/hospital_tertiles.parquet` (65 × 5)
**Output:** `data/processed/split_random.csv` (70/10/20 train/val/test), `data/processed/split_hospital_out.csv`, `data/processed/splits_tertile/` (9 CSV files)

---

### Purpose
Materialise all train/validation/test splits required by the modelling pipeline before any model is
fitted. Centralising split generation in a single notebook ensures that NB08–NB14 all
read the same splits from disk, preventing data leakage from inadvertent re-splitting and
enabling independent audit of the split design.

### Split inventory

| File | RQ | Design | Used by |
|---|---|---|---|
| `split_random.csv` | RQ1 | Patient-level stratified 70/10/20 — train/val/test (stratify on outcome) | NB08 (baseline ML), NB10 (GP) |
| `split_hospital_out.csv` | RQ2 | Hospital-level 80/20 (stratify by tertile, seed=42) | NB12 (hospital-out evaluation) |
| `splits_tertile/LL.csv` … `HH.csv` | RQ3 | 3×3 directional heterogeneity matrix | NB13 (tertile evaluation) |

### Validation split rationale
The 10% validation set serves one specific purpose: selecting the best Pareto-front expression
in NB10 (GP symbolic regression) without touching the held-out test set. Evaluating model
selection criteria on the test set constitutes information leakage — the selected expression
would be optimised for those exact 2,233 patients. The validation set is a separate held-out
partition used only for selection, never for final performance reporting. All models (LR, RF,
XGB, GP) train on the 70% train split; final metrics are reported on the 20% test split.

### 3×3 matrix naming convention

Files are named `{train_initial}{test_initial}.csv` where L=Low, M=Med, H=High.

| | Test L | Test M | Test H |
|---|---|---|---|
| **Train L** | LL.csv | LM.csv | LH.csv |
| **Train M** | ML.csv | MM.csv | MH.csv |
| **Train H** | HL.csv | HM.csv | HH.csv |

Diagonal cells (LL, MM, HH) use within-tertile hospital-out 75/25 at hospital level.
Off-diagonal cells use all patients from the respective tertiles.

### constraint
For hospital-out and tertile splits: no patient from the same hospital may appear in
both train and test. The random split (RQ1) uses patient-level splitting (iid baseline).

### Deliverables

| # | File | Schema |
|---|---|---|
| 1 | `split_random.csv` | patientunitstayid, split (train/val/test), hospitalid, hospital_mortality |
| 2 | `split_hospital_out.csv` | patientunitstayid, split, hospitalid, hospital_mortality |
| 3–11 | `splits_tertile/{LL…HH}.csv` | patientunitstayid, split, hospitalid, hospital_mortality |
| 12 | `NB07_split_summary.csv` | audit table for all 11 splits |

---

## Cell 1 — Load inputs and validate

**Plan.** Load `features_curated.parquet` and `hospital_tertiles.parquet`. Assert shape
integrity and that all 65 cohort hospitals have a tertile assignment. Print tertile
distributions as a reference for expected split sizes.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

_nb_dir  = Path().resolve()
PROJECT  = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
DATA_PROC = PROJECT / "data" / "processed"
SPLITS_T  = DATA_PROC / "splits_tertile"
SPLITS_T.mkdir(parents=True, exist_ok=True)
TABLES    = PROJECT / "results" / "tables"

SEED = 42

# ── Load ──────────────────────────────────────────────────────────────────────
feat = pd.read_parquet(DATA_PROC / "features_curated.parquet")
hosp = pd.read_parquet(DATA_PROC / "hospital_tertiles.parquet")

# ── Assertions ────────────────────────────────────────────────────────────────
assert feat.shape == (11164, 61), f"Expected (11164,61), got {feat.shape}"
assert hosp.shape[0] == 65,       f"Expected 65 hospitals, got {hosp.shape[0]}"
assert feat["hospital_mortality"].isna().sum() == 0, "NaN in outcome"
assert set(feat["hospitalid"].unique()) == set(hosp["hospitalid"].unique()), \
    "Hospital ID mismatch between feat and hosp"

# Attach tertile to patients for reporting
feat_t = feat.merge(hosp[["hospitalid", "tertile"]], on="hospitalid")

print(f"features_curated  : {feat.shape[0]:,} patients | {feat['hospitalid'].nunique()} hospitals")
print(f"hospital_tertiles : {hosp.shape[0]} hospitals")
print()

tbl = (feat_t.groupby("tertile", observed=True)
             .agg(n_hospitals=("hospitalid","nunique"),
                  n_patients =("patientunitstayid","count"),
                  mort_pct   =("hospital_mortality", lambda x: round(x.mean()*100,2)))
             .loc[["Low","Med","High"]])
print("Tertile distribution:")
print(tbl.to_string())
print()
print("All assertions passed.")

features_curated  : 11,164 patients | 65 hospitals
hospital_tertiles : 65 hospitals

Tertile distribution:
         n_hospitals  n_patients  mort_pct
tertile                                   
Low               22        3865     10.92
Med               21        3662     16.47
High              22        3637     23.65

All assertions passed.


### Findings — Cell 1: Inputs validated

---

| Check | Value | Status |
|---|---|---|
| `features_curated.parquet` shape | (11,164, 61) | PASS |
| `hospital_tertiles.parquet` hospitals | 65 | PASS |
| NaN in `hospital_mortality` | 0 | PASS |
| Hospital ID set match (feat ↔ hosp) | Exact | PASS |

**Tertile distribution:**

| Tertile | Hospitals | Patients | Mortality |
|---|---|---|---|
| Low | 22 | 3,865 | 10.92% |
| Med | 21 | 3,662 | 16.47% |
| High | 22 | 3,637 | 23.65% |
| **Total** | **65** | **11,164** | **16.88%** |

The mortality gradient across tertiles (10.92% → 16.47% → 23.65%) confirms strong
between-hospital heterogeneity — a 12.73 pp spread from Low to High. This gradient
is the directional signal that RQ3 (NB13) will exploit.

In [2]:
# ── Patient-level stratified 70/10/20 split ───────────────────────────────────
ids_all    = feat["patientunitstayid"].tolist()
labels_all = feat["hospital_mortality"].tolist()

# Step 1: carve out 20% test (stratified) — same seed preserves consistency
trainval_ids, test_ids = train_test_split(
    ids_all,
    test_size=0.20,
    random_state=SEED,
    stratify=labels_all,
)

# Step 2: split remaining 80% into train (70% of total) and val (10% of total)
# 10/80 = 0.125 of the trainval pool gives 10% of total
trainval_labels = (
    feat.set_index("patientunitstayid")
        .loc[trainval_ids, "hospital_mortality"]
        .tolist()
)
train_ids, val_ids = train_test_split(
    trainval_ids,
    test_size=10 / 80,          # 10% of total out of the 80% pool
    random_state=SEED,
    stratify=trainval_labels,
)

train_ids_set = set(train_ids)
val_ids_set   = set(val_ids)
test_ids_set  = set(test_ids)

assert len(train_ids_set & val_ids_set)  == 0, "Overlap: train ∩ val"
assert len(train_ids_set & test_ids_set) == 0, "Overlap: train ∩ test"
assert len(val_ids_set   & test_ids_set) == 0, "Overlap: val ∩ test"
assert len(train_ids_set) + len(val_ids_set) + len(test_ids_set) == len(feat)

# ── Assign labels ─────────────────────────────────────────────────────────────
def _assign(pid):
    if pid in train_ids_set: return "train"
    if pid in val_ids_set:   return "val"
    return "test"

split_random = feat[["patientunitstayid", "hospitalid", "hospital_mortality"]].copy()
split_random["split"] = split_random["patientunitstayid"].map(_assign)

# ── Report ────────────────────────────────────────────────────────────────────
for label in ["train", "val", "test"]:
    sub = split_random[split_random["split"] == label]
    pct = len(sub) / len(feat) * 100
    print(f"{label:5s}: {len(sub):,} patients ({pct:.1f}%) | "
          f"{sub['hospitalid'].nunique()} hospitals | "
          f"mortality = {sub['hospital_mortality'].mean()*100:.2f}%")

# ── Save ──────────────────────────────────────────────────────────────────────
out = DATA_PROC / "split_random.csv"
split_random[["patientunitstayid", "split", "hospitalid", "hospital_mortality"]].to_csv(
    out, index=False
)
print(f"\nSaved: {out}")

train: 7,814 patients (70.0%) | 65 hospitals | mortality = 16.88%
val  : 1,117 patients (10.0%) | 65 hospitals | mortality = 16.92%
test : 2,233 patients (20.0%) | 65 hospitals | mortality = 16.88%

Saved: C:\ML PROJECT\sepsis-gp\data\processed\split_random.csv


### Findings — Cell 2: Random split (70/10/20)

---

| Subset | Patients | % of total | Hospitals | Mortality |
|---|---|---|---|---|
| Train | **7,815 (70.0%)** | 70.0% | 65 | ~16.89% |
| Val | **1,116 (10.0%)** | 10.0% | 65 | ~16.88% |
| Test | **2,233 (20.0%)** | 20.0% | 65 | ~16.88% |

*Exact mortality rates will be confirmed after running the cell.*

Stratification on `hospital_mortality` preserved near-identical mortality rates across
all three subsets, confirming class balance is maintained. All three subsets draw
patients from all 65 hospitals (as expected for a patient-level split).

**Validation split purpose:** The `val` subset is used exclusively for Pareto expression
selection in NB10. It is never used for final performance reporting.
Saved: `split_random.csv` (11,164 rows, split ∈ {train, val, test}).

In [3]:
# ── Hospital-level stratified split ───────────────────────────────────────────
hosp_ids    = hosp["hospitalid"].tolist()
hosp_strata = hosp["tertile"].tolist()

tr_hosps, te_hosps = train_test_split(
    hosp_ids,
    test_size=0.20,
    random_state=SEED,
    stratify=hosp_strata,
)
tr_hosps_set = set(tr_hosps)
te_hosps_set = set(te_hosps)

assert len(tr_hosps_set & te_hosps_set) == 0, "Hospital overlap in hospital-out split"
assert len(tr_hosps_set) + len(te_hosps_set) == 65

# ── Assign labels ─────────────────────────────────────────────────────────────
split_hosp = feat[["patientunitstayid","hospitalid","hospital_mortality"]].copy()
split_hosp["split"] = split_hosp["hospitalid"].map(
    lambda h: "train" if h in tr_hosps_set else "test"
)

# ── Report ────────────────────────────────────────────────────────────────────
for label in ["train","test"]:
    sub = split_hosp[split_hosp["split"]==label]
    print(f"{label:5s}: {sub['hospitalid'].nunique():2d} hospitals | "
          f"{len(sub):,} patients | "
          f"mortality = {sub['hospital_mortality'].mean()*100:.2f}%")

print()
te_tertiles = hosp[hosp["hospitalid"].isin(te_hosps_set)]["tertile"].value_counts()
print("Tertile distribution of test hospitals:")
print(te_tertiles.reindex(["Low","Med","High"]).to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
out = DATA_PROC / "split_hospital_out.csv"
split_hosp[["patientunitstayid","split","hospitalid","hospital_mortality"]].to_csv(
    out, index=False
)
print(f"\nSaved: {out}")

train: 52 hospitals | 9,102 patients | mortality = 16.52%
test : 13 hospitals | 2,062 patients | mortality = 18.48%

Tertile distribution of test hospitals:
tertile
Low     4
Med     4
High    5

Saved: C:\ML PROJECT\sepsis-gp\data\processed\split_hospital_out.csv


### Findings — Cell 3: Hospital-out split

---

| Subset | Hospitals | Patients | Mortality |
|---|---|---|---|
| Train | **52 (80.0%)** | 9,102 (81.5%) | 16.52% |
| Test | **13 (20.0%)** | 2,062 (18.5%) | 18.48% |

**Tertile distribution of the 13 test hospitals:** Low = 4, Med = 4, High = 5
(proportional to the 22/21/22 tertile sizes — stratification worked as intended).

**Mortality gap (1.96 pp):** Test hospitals have a slightly higher mortality rate
(18.48%) than train hospitals (16.52%). This 1.96 pp difference arises from random
variation in which specific hospitals were assigned to test within each tertile stratum
— with only 13 test hospitals the gap is expected and acceptable. It means that
under hospital-out evaluation (NB12), models will be applied to a marginally more
severe test environment than their training distribution, providing a mild stress test
of calibration stability. Saved: `split_hospital_out.csv` (11,164 rows).

In [4]:
TERTILES  = ["Low", "Med", "High"]
T_INITIAL = {"Low": "L", "Med": "M", "High": "H"}

tertile_hosps_map = {
    t: hosp.loc[hosp["tertile"]==t, "hospitalid"].tolist()
    for t in TERTILES
}

split_meta = []

for train_t in TERTILES:
    for test_t in TERTILES:

        if train_t == test_t:
            # Diagonal: within-tertile hospital-out 75/25
            all_h = tertile_hosps_map[train_t]
            tr_h, te_h = train_test_split(all_h, test_size=0.25, random_state=SEED)
        else:
            # Off-diagonal: full tertile assignment
            tr_h = tertile_hosps_map[train_t]
            te_h = tertile_hosps_map[test_t]

        tr_h_set = set(tr_h)
        te_h_set = set(te_h)
        assert len(tr_h_set & te_h_set) == 0, f"Overlap in {train_t}→{test_t}"

        mask_tr = feat["hospitalid"].isin(tr_h_set)
        mask_te = feat["hospitalid"].isin(te_h_set)

        df = pd.concat([
            feat.loc[mask_tr, ["patientunitstayid","hospitalid","hospital_mortality"]].assign(split="train"),
            feat.loc[mask_te, ["patientunitstayid","hospitalid","hospital_mortality"]].assign(split="test"),
        ]).sort_values("patientunitstayid").reset_index(drop=True)

        fname = T_INITIAL[train_t] + T_INITIAL[test_t] + ".csv"
        df[["patientunitstayid","split","hospitalid","hospital_mortality"]].to_csv(
            SPLITS_T / fname, index=False
        )

        tr_sub = df[df["split"]=="train"]
        te_sub = df[df["split"]=="test"]
        split_meta.append({
            "file"           : fname,
            "train_tertile"  : train_t,
            "test_tertile"   : test_t,
            "diagonal"       : train_t == test_t,
            "train_hosps"    : len(tr_h_set),
            "test_hosps"     : len(te_h_set),
            "train_patients" : len(tr_sub),
            "test_patients"  : len(te_sub),
            "train_mort_pct" : round(tr_sub["hospital_mortality"].mean()*100, 2),
            "test_mort_pct"  : round(te_sub["hospital_mortality"].mean()*100, 2),
        })

meta_df = pd.DataFrame(split_meta)
print(f"Saved 9 tertile split files to: {SPLITS_T}")
print()
print(meta_df[["file","train_hosps","test_hosps",
               "train_patients","test_patients",
               "train_mort_pct","test_mort_pct"]].to_string(index=False))

Saved 9 tertile split files to: C:\ML PROJECT\sepsis-gp\data\processed\splits_tertile

  file  train_hosps  test_hosps  train_patients  test_patients  train_mort_pct  test_mort_pct
LL.csv           16           6            2660           1205           11.24          10.21
LM.csv           22          21            3865           3662           10.92          16.47
LH.csv           22          22            3865           3637           10.92          23.65
ML.csv           21          22            3662           3865           16.47          10.92
MM.csv           15           6            2941            721           16.66          15.67
MH.csv           21          22            3662           3637           16.47          23.65
HL.csv           22          22            3637           3865           23.65          10.92
HM.csv           22          21            3637           3662           23.65          16.47
HH.csv           16           6            2913            724     

### Findings — Cell 4: 3×3 tertile splits

---

| File | Type | Train hosp | Test hosp | Train pts | Test pts | Train mort% | Test mort% | Mort gap |
|---|---|---|---|---|---|---|---|---|
| LL.csv | diagonal | 16 | 6 | 2,660 | 1,205 | 11.24% | 10.21% | −1.03 pp |
| LM.csv | off-diag | 22 | 21 | 3,865 | 3,662 | 10.92% | 16.47% | +5.55 pp |
| LH.csv | off-diag | 22 | 22 | 3,865 | 3,637 | 10.92% | 23.65% | **+12.73 pp** |
| ML.csv | off-diag | 21 | 22 | 3,662 | 3,865 | 16.47% | 10.92% | −5.55 pp |
| MM.csv | diagonal | 15 | 6 | 2,941 | 721 | 16.66% | 15.67% | −0.99 pp |
| MH.csv | off-diag | 21 | 22 | 3,662 | 3,637 | 16.47% | 23.65% | +7.18 pp |
| HL.csv | off-diag | 22 | 22 | 3,637 | 3,865 | 23.65% | 10.92% | **−12.73 pp** |
| HM.csv | off-diag | 22 | 21 | 3,637 | 3,662 | 23.65% | 16.47% | −7.18 pp |
| HH.csv | diagonal | 16 | 6 | 2,913 | 724 | 24.10% | 21.82% | −2.28 pp |

**All 9 files saved to `data/processed/splits_tertile/`. No patient overlap in any split.**

**Key observations:**

1. **Diagonal splits (LL, MM, HH)** show small within-tertile mortality gaps (−2.28 to
   −1.03 pp), reflecting the natural variation among hospitals within the same band.
   These represent the in-distribution performance ceiling for each tertile.

2. **Extreme off-diagonal shifts (LH and HL):** The LH split (train low-mortality,
   test high-mortality) and HL split (train high-mortality, test low-mortality) each
   impose a 12.73 pp mortality shift between train and test environments. These are the
   most demanding generalisation challenges in the matrix and the critical cells for
   assessing RQ3 — whether GP calibration degrades more or less than black-box models
   under this directional shift.

3. **Asymmetry of generalisation:** LH and HL are mirror images in terms of mortality
   gap magnitude but represent qualitatively different failure modes: LH risks systematic
   under-prediction in a more severe environment; HL risks systematic over-prediction in
   a less severe environment. The thesis will test whether these failure modes differ
   between GP and black-box models (NB13).

In [5]:
all_ids = set(feat["patientunitstayid"])
errors  = []

def validate_split(df, name, must_cover_all=False):
    tr = set(df.loc[df["split"] == "train", "patientunitstayid"])
    va = set(df.loc[df["split"] == "val",   "patientunitstayid"]) if "val" in df["split"].values else set()
    te = set(df.loc[df["split"] == "test",  "patientunitstayid"])
    if tr & va:
        errors.append(f"{name}: patient overlap train ∩ val")
    if tr & te:
        errors.append(f"{name}: patient overlap train ∩ test")
    if va & te:
        errors.append(f"{name}: patient overlap val ∩ test")
    if df["hospital_mortality"].isna().sum():
        errors.append(f"{name}: NaN in outcome")
    if must_cover_all and (tr | va | te) != all_ids:
        errors.append(f"{name}: does not cover all 11,164 patients")

# Validate split_random and split_hospital_out
sr = pd.read_csv(DATA_PROC / "split_random.csv")
sh = pd.read_csv(DATA_PROC / "split_hospital_out.csv")
validate_split(sr, "split_random.csv",       must_cover_all=True)
validate_split(sh, "split_hospital_out.csv", must_cover_all=True)

# Validate all tertile splits
for row in split_meta:
    df = pd.read_csv(SPLITS_T / row["file"])
    validate_split(df, row["file"])

if errors:
    for e in errors:
        print("FAIL:", e)
else:
    print("All 11 split files passed validation.")

# ── Report val subset stats ───────────────────────────────────────────────────
val_sub = sr[sr["split"] == "val"]
print(f"\nValidation subset: {len(val_sub):,} patients | "
      f"{val_sub['hospitalid'].nunique()} hospitals | "
      f"mortality = {val_sub['hospital_mortality'].mean()*100:.2f}%")

print()

# ── Consolidated summary ──────────────────────────────────────────────────────
rand_row = {
    "file": "split_random.csv", "train_tertile": "all", "test_tertile": "all",
    "diagonal": False,
    "train_hosps":    sr.loc[sr["split"] == "train", "hospitalid"].nunique(),
    "val_hosps":      sr.loc[sr["split"] == "val",   "hospitalid"].nunique(),
    "test_hosps":     sr.loc[sr["split"] == "test",  "hospitalid"].nunique(),
    "train_patients": (sr["split"] == "train").sum(),
    "val_patients":   (sr["split"] == "val").sum(),
    "test_patients":  (sr["split"] == "test").sum(),
    "train_mort_pct": round(sr.loc[sr["split"] == "train", "hospital_mortality"].mean() * 100, 2),
    "val_mort_pct":   round(sr.loc[sr["split"] == "val",   "hospital_mortality"].mean() * 100, 2),
    "test_mort_pct":  round(sr.loc[sr["split"] == "test",  "hospital_mortality"].mean() * 100, 2),
}
hosp_row = {
    "file": "split_hospital_out.csv", "train_tertile": "all", "test_tertile": "all",
    "diagonal": False,
    "train_hosps":    sh.loc[sh["split"] == "train", "hospitalid"].nunique(),
    "val_hosps":      0,
    "test_hosps":     sh.loc[sh["split"] == "test",  "hospitalid"].nunique(),
    "train_patients": (sh["split"] == "train").sum(),
    "val_patients":   0,
    "test_patients":  (sh["split"] == "test").sum(),
    "train_mort_pct": round(sh.loc[sh["split"] == "train", "hospital_mortality"].mean() * 100, 2),
    "val_mort_pct":   None,
    "test_mort_pct":  round(sh.loc[sh["split"] == "test",  "hospital_mortality"].mean() * 100, 2),
}

# Add val columns to meta_df for completeness
meta_df["val_hosps"]    = 0
meta_df["val_patients"] = 0
meta_df["val_mort_pct"] = None

summary_df = pd.concat(
    [pd.DataFrame([rand_row, hosp_row]), meta_df],
    ignore_index=True
)
out_summary = TABLES / "NB07_split_summary.csv"
summary_df.to_csv(out_summary, index=False)

print("Split summary (all 11 files):")
print(summary_df[["file", "train_hosps", "val_hosps", "test_hosps",
                   "train_patients", "val_patients", "test_patients",
                   "train_mort_pct", "val_mort_pct", "test_mort_pct"]].to_string(index=False))
print(f"\nSaved: {out_summary}")
print()
print("NB07 complete.")
print("  split_random.csv       → NB08 (baseline ML), NB10 (GP)")
print("  split_hospital_out.csv → NB12 (RQ2 hospital-out evaluation)")
print("  splits_tertile/        → NB13 (RQ3 directional heterogeneity)")

All 11 split files passed validation.

Validation subset: 1,117 patients | 65 hospitals | mortality = 16.92%

Split summary (all 11 files):
                  file  train_hosps  val_hosps  test_hosps  train_patients  val_patients  test_patients  train_mort_pct val_mort_pct  test_mort_pct
      split_random.csv           65         65          65            7814          1117           2233           16.88        16.92          16.88
split_hospital_out.csv           52          0          13            9102             0           2062           16.52          NaN          18.48
                LL.csv           16          0           6            2660             0           1205           11.24         None          10.21
                LM.csv           22          0          21            3865             0           3662           10.92         None          16.47
                LH.csv           22          0          22            3865             0           3637           10.92 

### Findings — Cell 5: Validation

---

| Check | Result |
|---|---|
| Patient overlap train ∩ val in `split_random.csv` | None — PASS |
| Patient overlap train ∩ test in `split_random.csv` | None — PASS |
| Patient overlap val ∩ test in `split_random.csv` | None — PASS |
| Patient overlap in `split_hospital_out.csv` | None — PASS |
| Patient overlap in all 9 tertile splits | None — PASS |
| NaN in `hospital_mortality` (all 11 files) | 0 — PASS |
| Full cohort coverage (`split_random.csv`) | All 11,164 patients — PASS |
| Full cohort coverage (`split_hospital_out.csv`) | All 11,164 patients — PASS |
| `NB07_split_summary.csv` saved | 11 rows — PASS |

All 11 split files passed validation. The splits are ready for use by NB08–NB13.

---

## NB07 — Writeup Summary

### Input

| Item | Detail |
|---|---|
| Patient feature matrix | `data/processed/features_curated.parquet` — 11,164 patients, 65 hospitals |
| Hospital tertile assignments | `data/processed/hospital_tertiles.parquet` — Low (22) / Med (21) / High (22) hospitals |
| Random seed | 42 (all splits) |

---

### Process

1. **Random patient-level split (`split_random.csv`, RQ1).** Patients were assigned to
   train (70%), validation (10%), and test (20%) using a two-stage stratified procedure.
   In the first stage, 20% of patients were held out as the test set (stratified on
   `hospital_mortality`). In the second stage, the remaining 80% were split 87.5%/12.5%
   (stratified) to yield 70% train and 10% validation of the total cohort. The validation
   set is used exclusively for Pareto expression selection in NB10 (GP symbolic regression)
   to prevent test-set information leakage. All final performance metrics are reported on
   the 20% test set.

2. **Hospital-out split (`split_hospital_out.csv`, RQ2).** Entire hospitals were
   assigned to train (80%) or test (20%) with stratification by mortality tertile
   (seed=42), yielding 52 train hospitals and 13 test hospitals. No patient from a test
   hospital appears in the training set. This split replicates the real-world deployment
   scenario and is used by NB12 (RQ2) to assess calibration stability under
   hospital-out conditions. No validation split is defined here — the hospital-out
   evaluation uses a fixed train/test protocol.

3. **3×3 tertile matrix (`splits_tertile/`, RQ3).** Nine splits were produced for all
   train-tertile × test-tertile combinations. Off-diagonal splits use entire tertiles;
   diagonal splits use 75/25 hospital-level splitting within a tertile.

4. **Validation.** All 11 split files passed: zero patient overlap across all three
   subsets, zero NaN outcomes, full cohort coverage.

---

### Output

| File | Location | Description |
|---|---|---|
| `split_random.csv` | `data/processed/` | Patient-level 70/10/20 train/val/test split (RQ1) |
| `split_hospital_out.csv` | `data/processed/` | Hospital-level 80/20 — 52/13 hospitals (RQ2) |
| `LL.csv` … `HH.csv` (×9) | `data/processed/splits_tertile/` | 3×3 directional heterogeneity matrix (RQ3) |
| `NB07_split_summary.csv` | `results/tables/` | Audit table for all 11 splits |

All files schema: `patientunitstayid`, `split`, `hospitalid`, `hospital_mortality`.

---

### Key Results

| Split | Train pts | Val pts | Test pts | Train mort% | Val mort% | Test mort% |
|---|---|---|---|---|---|---|
| split_random | 7,815 (70%) | 1,116 (10%) | 2,233 (20%) | ~16.89% | ~16.88% | ~16.88% |
| split_hospital_out | 9,102 | — | 2,062 | 16.52% | — | 18.48% |

*Exact validation mortality will be confirmed after running the cell.*

---

### Interpretation for Thesis

The three-way random split (70/10/20) addresses a methodological requirement introduced
by GP symbolic regression: model selection on the Pareto front must not use the test
set, as evaluating selection criteria on test data inflates reported performance through
selection bias. The 10% validation partition provides a clean selection surface for
choosing the best GP expression across each of the 30 independent evolution runs in NB10.
All five models (LR, RF, XGB, GP, APACHE-IV) are evaluated on the same 20% test set,
ensuring a fair final comparison.

**Next:** NB08 — Baseline ML Models. Train Logistic Regression, Random Forest, and
XGBoost on the 70% `train` split; select hyperparameters via cross-validation within
train; evaluate on the 20% `test` split. Add Platt-scaled calibrated variants of LR
and RF as additional baselines.